In [1]:
%pip install anoexpress -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 38.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.4/204.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.9/25.9 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 M

In [2]:
import anoexpress as xpress
import numpy as np
import pandas as pd

### Perform GSEA on custom data

First, a file of AGAP GeneIDs must be uploaded into Colab. It must have the GeneIDs in the first column, with no header. Click on the files tab of Colab on the left hand side, and use the upload button to upload your file. It can be a .txt, .csv, .tsv or .xlsx file.

Then pass the file path of your GeneID file to the `gene_ids` parameter of `xpress.go_hypergeometric()`.

In [44]:
import glob

In [45]:
# glob all csvs in wd and split by _ to get comparisons

csvs = glob.glob('*.csv')
csvs

['coluzziiCont_coluzziiPM.csv',
 'Kisumu_gambiaeCont.csv',
 'Ngousso_coluzziiCont.csv',
 'gambiaeCont_gambiaePM.csv']

In [49]:
comps = np.array([csv.rstrip(".csv").split('_') for csv in csvs])[[0,3]]
comps

array([['coluzziiCont', 'coluzziiPM'],
       ['gambiaeCont', 'gambiaePM']], dtype='<U12')

In [55]:
df_gsea = {}
for comp in comps:
    df = pd.read_csv(f'{comp[0]}_{comp[1]}.csv')
    upgenes = df.query("padj < 0.05 and FC > 1.5")
#    downgenes = df.query("padj < 0.05 and log2fc < 0")

    df_gsea[f'{comp[0]}_{comp[1]}_go'] = xpress.go_hypergeometric(analysis='gamb_colu', gene_ids=upgenes['GeneID'].tolist())
    df_gsea[f'{comp[0]}_{comp[1]}_pfam'] = xpress.pfam_hypergeometric(analysis='gamb_colu', gene_ids=upgenes['GeneID'].tolist())
    df_gsea[f'{comp[0]}_{comp[1]}_kegg'] = xpress.kegg_hypergeometric(analysis='gamb_colu', gene_ids=upgenes['GeneID'].tolist())

100%|██████████| 141/141 [00:00<00:00, 277.96it/s]


In [ ]:
%pip install xlsx

In [62]:

data = df_gsea
output_file = "gsea-bouake.xlsx"

# Write the DataFrames to the Excel file
with pd.ExcelWriter(output_file, engine=None) as writer:
    for sheet_name, df in data.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"DataFrames have been written to {output_file}")

DataFrames have been written to gsea-bouake.xlsx


In [59]:
for comp in comps:
    print(comp)
    print(df_gsea[f'{comp[0]}_{comp[1]}_kegg'].query("padj < 0.05").to_markdown())
    print("\n")

['coluzziiCont' 'coluzziiPM']
| annotation   | pval   | padj   | description   |
|--------------|--------|--------|---------------|


['gambiaeCont' 'gambiaePM']
|    | annotation   |        pval |        padj | description                                 |
|---:|:-------------|------------:|------------:|:--------------------------------------------|
|  0 | aga04141     | 2.30947e-06 | 0.000325635 | Protein processing in endoplasmic reticulum |
|  1 | aga02010     | 2.18314e-05 | 0.00153911  | ABC transporters                            |
|  2 | aga04145     | 0.000309703 | 0.014556    | Phagosome                                   |
|  3 | aga04214     | 0.00112246  | 0.0395666   | Apoptosis - fly                             |




In [ ]:
# gene_id_file = 'my_genes.txt' # replace with your gene ID file name

# my_genes_go_enrichment = xpress.go_hypergeometric(analysis='gamb_colu_arab_fun', gene_ids=gene_id_file) #ignore the analysis argument, it wont be used in this case
# my_genes_go_enrichment

### GO annotations on AnoExpress candidates

In [ ]:
median_go_hypergeo_df = xpress.go_hypergeometric(analysis="gamb_colu_arab_fun", func=np.nanmedian)
median_go_hypergeo_df.query("padj < 0.05")

# Pfam Domains

In [ ]:
median_pfam_hypergeo_df = xpress.pfam_hypergeometric(analysis="gamb_colu_arab_fun", func=np.nanmedian)
median_pfam_hypergeo_df.head(30)

Lots of cool stuff here too - P450s, obps, ORs, GRs, GST, Elongases (ELO), chitin binding seems to be coming up alot.

In [ ]:
### FOR REFERENCE - TO MAP pfam domain to gene family name. Lots of other stuff coming up as significant I have no idea what that is
pfam_domain_names = {'Cytochrome P450s':'p450',
             'GSTs':['GST_N', 'GST_N_3', 'GST_C'],
             'Carboxylesterases': 'COesterase',
             'ABC-transporters':['ABC_membrane', 'ABC_tran'],
             'CSP': 'OS-D',
             'UGTs': 'UDPGT',
             'Odorant binding proteins':'PBP_GOBP',
             'Olfactory receptors':'7tm_6',
             'Ionotropic receptors':['Lig_chan','7tm_1'],
             'Gustatory receptors': '7tm_7',
             'Fatty acid synthases':'ketoacyl-synt',
             'FA Elongase':'ELO',
             'FA desaturase':'FA_desaturase',
             'FA reductase':'NAD_binding_4',
             }

## Kegg pathways

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", 500)

In [ ]:
median_kegg_hypergeo_df = xpress.kegg_hypergeometric(analysis="gamb_colu_arab_fun", func=np.nanmedian)
median_kegg_hypergeo_df.head(50)